# Aula 2 - Notebook 2
## Investigacao Multiagente com CrewAI
### Papeis, tarefas e tools decididas pelos agentes

Neste notebook, construimos um sistema multiagente usando CrewAI para entender o raciocinio de construcao passo a passo.

Objetivo principal:
- transformar um fluxo manual de investigacao em uma colaboracao estruturada entre agentes especializados;
- manter rastreabilidade das decisoes (quais evidencias foram consultadas e por que);
- preparar uma base segura para evolucao futura (por exemplo, geracao de patch).

Como pensar o design antes do codigo:
1. Problema: um incidente precisa ser investigado com evidencias e criterios de revisao.
2. Decomposicao: dividir o trabalho em papeis com responsabilidades claras.
3. Ferramentas: dar aos agentes acesso a leitura/busca/testes, sem permitir escrita automatica.
4. Orquestracao: executar tarefas em sequencia, preservando contexto entre etapas.
5. Governanca: revisar a proposta final para bloquear conclusoes fracas ou sem testes.

Comparacao conceitual entre frameworks:
- Em CrewAI, modelamos uma equipe (papeis + tarefas + processo).
- Em LangGraph, modelamos explicitamente um grafo de estado (nos + transicoes + estado).
- Nenhum e "melhor" em absoluto: a escolha depende de clareza, controle de fluxo e nivel de personalizacao necessario.

Resultado esperado ao final: um artefato estruturado com hipotese, mapeamento de codigo, proposta conceitual e revisao critica, sem modificar o projeto automaticamente.

In [1]:
from pathlib import Path
import os
import json
import textwrap
import subprocess
import sys
from datetime import datetime

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TARGET_PROJECT_DIR = PROJECT_ROOT / "target_project/mini_orders_pipeline"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXTENSOES_INDEXADAS = {".md", ".txt", ".log", ".py", ".json", ".toml"}


def ler_texto(caminho):
    return Path(caminho).read_text(encoding="utf-8")


def listar_arquivos_recursivamente(raiz, extensoes=EXTENSOES_INDEXADAS):
    arquivos = []
    for caminho in Path(raiz).rglob("*"):
        if caminho.is_file() and caminho.suffix in extensoes:
            arquivos.append(caminho)
    return sorted(arquivos)


def montar_corpus(data_dir=DATA_DIR, target_project_dir=TARGET_PROJECT_DIR):
    arquivos = listar_arquivos_recursivamente(data_dir) + listar_arquivos_recursivamente(target_project_dir)
    documentos = []
    for caminho in arquivos:
        relativo = caminho.relative_to(PROJECT_ROOT).as_posix()
        documentos.append({
            "path": relativo,
            "content": ler_texto(caminho),
            "source_type": "target_project" if "target_project" in relativo else "evidence",
        })
    return documentos


def criar_indice_tfidf(documentos):
    vectorizer = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2))
    matriz = vectorizer.fit_transform([doc["content"] for doc in documentos])
    return {"vectorizer": vectorizer, "matrix": matriz, "documents": documentos}


def buscar_no_indice(indice, consulta, k=5):
    consulta_vetor = indice["vectorizer"].transform([consulta])
    scores = cosine_similarity(consulta_vetor, indice["matrix"])[0]
    ordenados = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
    resultados = []
    for idx, score in ordenados:
        doc = indice["documents"][idx]
        resultados.append({
            "path": doc["path"],
            "score": round(float(score), 4),
            "source_type": doc["source_type"],
            "preview": doc["content"][:1200],
        })
    return resultados


def listar_tickets():
    return [p.relative_to(PROJECT_ROOT).as_posix() for p in sorted((DATA_DIR / "tickets").glob("*.md"))]


def carregar_arquivo_relativo(caminho_relativo):
    caminho = PROJECT_ROOT / caminho_relativo
    if not caminho.exists():
        return json.dumps({"erro": f"Arquivo não encontrado: {caminho_relativo}"}, ensure_ascii=False)
    return caminho.read_text(encoding="utf-8")


def listar_arquivos_do_target_project():
    arquivos = listar_arquivos_recursivamente(TARGET_PROJECT_DIR)
    return [p.relative_to(PROJECT_ROOT).as_posix() for p in arquivos]


def executar_testes_target_project():
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=TARGET_PROJECT_DIR,
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        "returncode": resultado.returncode,
        "stdout": resultado.stdout,
        "stderr": resultado.stderr,
    }


def salvar_json_saida(nome_arquivo, conteudo):
    caminho = OUTPUT_DIR / nome_arquivo
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(conteudo, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho.relative_to(PROJECT_ROOT).as_posix()

DOCUMENTOS = montar_corpus()
INDICE = criar_indice_tfidf(DOCUMENTOS)
print(f"Documentos indexados: {len(DOCUMENTOS)}")

Documentos indexados: 11


## Tools para a crew

As funções abaixo são expostas como tools para os agentes. Nesta etapa, a decisão de design é importante:
- os agentes precisam de autonomia para investigar;
- mas essa autonomia deve ser limitada a operações seguras de leitura/inspeção.

Raciocínio por trás das tools escolhidas:
1. `buscar_evidencias`: reduz espaço de busca e acelera hipóteses iniciais.
2. `ler_arquivo`: permite validação direta de trechos e evita suposições.
3. `listar_arquivos_do_target_project`: ajuda o analista a mapear impacto real no código.
4. `executar_testes_target_project`: adiciona verificação objetiva para sustentar ou refutar hipóteses.

Princípio: toda afirmação relevante deve poder ser vinculada a evidência observável (arquivo, ticket, log ou resultado de teste).

Limite de segurança adotado neste notebook:
- não há tool de escrita/edição de arquivos;
- portanto, o sistema é investigativo e propositivo, e não executa mudanças automáticas.

No framework CrewAI, utilizamos o decorador `@tool` para identificar apropriadamente as ferramentas.

In [4]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool


def exigir_openai_api_key():
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Defina OPENAI_API_KEY para executar os agentes reais com CrewAI.")


@tool("buscar_evidencias")
def crew_buscar_evidencias(consulta: str) -> str:
    """Busca evidências em tickets, logs, runbooks, histórico, políticas e código do target_project."""
    return json.dumps(buscar_no_indice(INDICE, consulta, 6), ensure_ascii=False, indent=2)


@tool("ler_arquivo")
def crew_ler_arquivo(caminho_relativo: str) -> str:
    """Lê um arquivo pelo caminho relativo ao projeto do curso."""
    return carregar_arquivo_relativo(caminho_relativo)


@tool("listar_arquivos_do_target_project")
def crew_listar_arquivos_do_target_project() -> str:
    """Lista arquivos disponíveis no target_project."""
    return json.dumps(listar_arquivos_do_target_project(), ensure_ascii=False, indent=2)


@tool("executar_testes_target_project")
def crew_executar_testes_target_project() -> str:
    """Executa a suíte de testes do target_project."""
    return json.dumps(executar_testes_target_project(), ensure_ascii=False, indent=2)


TOOLS_CREW = [
    crew_buscar_evidencias,
    crew_ler_arquivo,
    crew_listar_arquivos_do_target_project,
    crew_executar_testes_target_project,
]


def criar_agente_crewai(role, goal, backstory):
    exigir_openai_api_key()
    return Agent(
        role=role,
        goal=goal,
        backstory=backstory,
        tools=TOOLS_CREW,
        verbose=True,
        allow_delegation=False,
    )


def criar_task(descricao, agente, saida_esperada):
    return Task(
        description=descricao,
        agent=agente,
        expected_output=saida_esperada,
    )

## Definindo agentes por papéis

Nesta etapa, modelamos uma "equipe técnica" com especialização funcional. A ideia é reduzir ambiguidade e aumentar qualidade das saídas.

A criação de papeis especializados, decorrente do princípio de passar o sistema para uma abordagem de múltiplos agentes, possibilita os pontos abaixo.
- foco: cada agente concentra esforço em um tipo de decisão;
- qualidade: prompts mais específicos tendem a reduzir respostas genéricas;
- auditabilidade: fica mais claro em qual etapa uma decisão ruim surgiu.

Função de cada papel neste notebook:
1. Investigador: constrói hipótese inicial com base em evidências.
2. Analista de código: traduz hipótese em impacto técnico no projeto-alvo.
3. Propositor: descreve correção conceitual e plano de testes.
4. Revisor: valida consistência, riscos e se deve avançar.

Boas práticas ao desenhar papéis:
- objetivos curtos e verificáveis;
- backstory suficiente para orientar estilo de raciocínio, sem exagero narrativo;
- evitar sobreposição excessiva entre agentes para não duplicar trabalho.

In [5]:
investigador = criar_agente_crewai(
    role="Investigador técnico de incidentes",
    goal="Encontrar evidências relevantes e formular uma hipótese sustentada sobre o incidente.",
    backstory="Você investiga falhas em pipelines de dados usando logs, tickets, runbooks, histórico e políticas."
)

analista_codigo = criar_agente_crewai(
    role="Analista de código do target_project",
    goal="Relacionar o incidente a arquivos e funções do projeto-alvo.",
    backstory="Você lê código Python e identifica pontos de fragilidade associados a incidentes reportados."
)

propositor = criar_agente_crewai(
    role="Propositor de correção conceitual",
    goal="Propor uma alteração segura, revisável e acompanhada de plano de testes.",
    backstory="Você não altera código automaticamente. Sua função é preparar uma proposta para revisão humana."
)

revisor = criar_agente_crewai(
    role="Revisor de segurança e limites de autonomia",
    goal="Verificar se a proposta está sustentada por evidências e se pode avançar para preparação de patch.",
    backstory="Você bloqueia propostas genéricas, não rastreáveis ou sem plano de testes."
)

## Definindo tarefas

As tarefas formalizam a passagem de contexto entre agentes e transformam papéis em execução concreta.

Raciocínio da sequência adotada:
1. Investigar primeiro: sem hipótese inicial, o restante vira especulação.
2. Mapear código depois: conecta hipótese ao sistema real.
3. Propor correção em seguida: somente após entender causa e impacto.
4. Revisar por último: etapa de controle de qualidade e segurança.

Como ler uma boa descrição de tarefa:
- contexto claro (qual incidente e com quais insumos);
- ação esperada (o que precisa ser analisado);
- restrições (ex.: não alterar código);
- saída esperada verificável (formato/critério de qualidade).

In [6]:
incident_id = "INC-002"

tarefa_investigacao = criar_task(
    descricao=f"""
Investigue o incidente {incident_id}.
Use tools para buscar e ler tickets, logs, runbooks, histórico e políticas.
Produza causa provável, evidências consultadas e confiança do diagnóstico.
""",
    agente=investigador,
    saida_esperada="Diagnóstico técnico com causa provável, evidências e confiança."
)

tarefa_codigo = criar_task(
    descricao=f"""
A partir da investigação do incidente {incident_id}, identifique arquivos e funções do target_project que podem estar envolvidos.
Use tools para listar e ler arquivos do target_project.
""",
    agente=analista_codigo,
    saida_esperada="Lista justificada de módulos afetados no target_project."
)

tarefa_proposta = criar_task(
    descricao=f"""
Produza uma proposta conceitual de correção para o incidente {incident_id}.
Inclua plano de testes. Não altere código e não gere PR.
""",
    agente=propositor,
    saida_esperada="Proposta conceitual de correção com plano de testes."
)

tarefa_revisao = criar_task(
    descricao=f"""
Revise a proposta final para o incidente {incident_id}.
Decida se ela pode seguir para preparação de patch.
Explique riscos, lacunas e necessidade de revisão humana.
""",
    agente=revisor,
    saida_esperada="Revisão final com aprovação, bloqueio ou condições para avançar."
)

## Executando a crew

Nesta etapa, as peças anteriores viram execução coordenada.

O que acontece no `kickoff()` em alto nível:
- a crew inicia no processo sequencial definido;
- cada tarefa é atribuída ao agente correspondente;
- o agente decide quando usar tools para coletar evidências;
- a saída de uma etapa alimenta o contexto da etapa seguinte.

Como interpretar os logs verbose:
1. verifique quais tools foram chamadas e com quais consultas;
2. observe se as conclusões citam evidências concretas;
3. identifique saltos lógicos (conclusão forte com evidência fraca);
4. registre pontos de melhoria em prompt, tarefa ou ordem do processo.

Critério de qualidade da execução: rastreabilidade. Se não for possível explicar "de onde veio" cada conclusão, a pipeline precisa de ajuste.

In [7]:
crew = Crew(
    agents=[investigador, analista_codigo, propositor, revisor],
    tasks=[tarefa_investigacao, tarefa_codigo, tarefa_proposta, tarefa_revisao],
    process=Process.sequential,
    verbose=True,
)

resultado = crew.kickoff()
print(resultado)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 28d95834-4d12-4e33-b317-34e39f933576                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Investigue o incidente INC-002.                                                                                │
│  Use tools para buscar e ler tickets, logs, runbooks, histórico e políticas.                                    │
│  Produza causa provável, evidências consultadas e confiança do diagnóstico.                                     │
│                                                                                                                 │
│  ID: e03ec0c4-cb92-4b81-aa90-4c61292e031f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador técnico de incidentes                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Investigue o incidente INC-002.                                                                                │
│  Use tools para buscar e ler tickets, logs, runbooks, histórico e políticas.                                    │
│  Produza causa provável, evidências consultadas e confiança do diagnóstico.                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: buscar_evidencias                                                                                        │
│  Args: {'consulta': 'INC-002'}                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor de segurança e limites de autonomia                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Proposta Final para o Incidente INC-002 - Revisão de Segurança e Limites de Autonomia                          │
│                                                                                                                 │
│  Após análise detalhada da proposta conceitual para correção do incidente INC-002, realizo a seguinte           │
│  avaliação:                                                                                                     │
│                                                                                                                 │
│  1. Sustentação da Proposta                                                                                     │
│  A proposta está bem fundamentada no diagnóstico técnico apresentado, que tem alta confiança. Os problemas      │
│  causados pela formatação incorreta do campo `order_total` são claramente apresentados, assim como as           │
│  consequências no cálculo das métricas.                                                                         │
│                                                                                                                 │
│  2. Especificidade e Plano de Testes                                                                            │
│  A proposta inclui um plano de testes sólido e detalhado que cobre:                                             │
│                                                                                                                 │
│  - Testes unitários para diferentes formatos de entrada, incluindo casos válidos e inválidos.                   │
│  - Testes de integração para validação da transformação dentro do pipeline e registro de alertas.               │
│  - Testes de aceitação para garantir que os dados corrigidos reflitam corretamente na métrica do dashboard sem  │
│  impactar negativamente a operação.                                                                             │
│  - Testes de regressão para garantir que funcionalidades relacionadas não sejam afetadas.                       │
│                                                                                                                 │
│  Este abrangente plano de testes atende a necessidade de validação rigorosa da correção.                        │
│                                                                                                                 │
│  3. Lacunas e Riscos Identificados                                                                              │
│                                                                                                                 │
│  - O documento indica que não há arquivos disponíveis para análise no projeto target_project, o que impede a    │
│  identificação precisa dos módulos ou pontos de inserção do código de correção — implicando risco de            │
│  implementação sem contexto do código existente.                                                                │
│  - Como o parceiro `mobile_app` está envolvido, há risco de recorrência do problema caso não haja efetiva       │
│  comunicação e alinhamento com ele, o que foi contemplado na proposta como ação paralela, mas depende de        │
│  colaboração externa.                                                                                           │
│  - A proposta menciona o registro de logs e alertas, po

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Revise a proposta final para o incidente INC-002.                                                              │
│  Decida se ela pode seguir para preparação de patch.                                                            │
│  Explique riscos, lacunas e necessidade de revisão humana.                                                      │
│                                                                                                                 │
│  Agent: Revisor de segurança e limites de autonomia                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Proposta Final para o Incidente INC-002 - Revisão de Segurança e Limites de Autonomia

Após análise detalhada da proposta conceitual para correção do incidente INC-002, realizo a seguinte avaliação:

1. Sustentação da Proposta
A proposta está bem fundamentada no diagnóstico técnico apresentado, que tem alta confiança. Os problemas causados pela formatação incorreta do campo `order_total` são claramente apresentados, assim como as consequências no cálculo das métricas.

2. Especificidade e Plano de Testes
A proposta inclui um plano de testes sólido e detalhado que cobre:

- Testes unitários para diferentes formatos de entrada, incluindo casos válidos e inválidos.
- Testes de integração para validação da transformação dentro do pipeline e registro de alertas.
- Testes de aceitação para garantir que os dados corrigidos reflitam corretamente na métrica do dashboard sem impactar negativamente a operação.
- Testes de regressão para garantir que funcionalidades relacionadas não sejam afetadas

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 28d95834-4d12-4e33-b317-34e39f933576                                                                       │
│  Final Output: Proposta Final para o Incidente INC-002 - Revisão de Segurança e Limites de Autonomia            │
│                                                                                                                 │
│  Após análise detalhada da proposta conceitual para correção do incidente INC-002, realizo a seguinte           │
│  avaliação:                                                                                                     │
│                                                                                                                 │
│  1. Sustentação da Proposta                                                                                     │
│  A proposta está bem fundamentada no diagnóstico técnico apresentado, que tem alta confiança. Os problemas      │
│  causados pela formatação incorreta do campo `order_total` são claramente apresentados, assim como as           │
│  consequências no cálculo das métricas.                                                                         │
│                                                                                                                 │
│  2. Especificidade e Plano de Testes                                                                            │
│  A proposta inclui um plano de testes sólido e detalhado que cobre:                                             │
│                                                                                                                 │
│  - Testes unitários para diferentes formatos de entrada, incluindo casos válidos e inválidos.                   │
│  - Testes de integração para validação da transformação dentro do pipeline e registro de alertas.               │
│  - Testes de aceitação para garantir que os dados corrigidos reflitam corretamente na métrica do dashboard sem  │
│  impactar negativamente a operação.                                                                             │
│  - Testes de regressão para garantir que funcionalidades relacionadas não sejam afetadas.                       │
│                                                                                                                 │
│  Este abrangente plano de testes atende a necessidade de validação rigorosa da correção.                        │
│                                                                                                                 │
│  3. Lacunas e Riscos Identificados                                                                              │
│                                                                                                                 │
│  - O documento indica que não há arquivos disponíveis para análise no projeto target_project, o que impede a    │
│  identificação precisa dos módulos ou pontos de inserção do código de correção — implicando risco de            │
│  implementação sem contexto do código existente.                                                                │
│  - Como o parceiro `mobile_app` está envolvido, há risco de recorrência do problema caso não haja efetiva       │
│  comunicação e alinhamento com ele, o que foi contemplado na proposta como ação paralela, mas depende de        │
│  colaboração externa.                                                                                           │
│  - A proposta menciona o registro de logs e alertas, p

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Salvando o resultado

Apos a execucao, salvamos um artefato persistente para revisao humana e comparacao entre abordagens.

Por que salvar em JSON textual neste momento:
- simplicidade de inspecao;
- facilidade de versionamento e comparacao entre execucoes;
- independencia de estruturas internas de objetos do framework.

Boa pratica de governanca:
- incluir `incident_id` e `framework` para rastreio;
- registrar observacao explicita de que nao houve alteracao automatica de codigo;
- manter outputs em pasta previsivel para auditoria posterior.

In [8]:
resultado_crewai = {
    "incident_id": incident_id,
    "framework": "CrewAI",
    "resultado": str(resultado),
    "observacao": "Nenhum arquivo foi alterado. A crew produziu uma proposta para revisão humana."
}

caminho = salvar_json_saida("reports/incidente_inc002_crewai_multiagentes.json", resultado_crewai)
print("Arquivo salvo em:", caminho)

Arquivo salvo em: outputs/reports/incidente_inc002_crewai_multiagentes.json


## Conclusão

Este notebook mostrou uma forma pratica de estruturar investigacao de incidentes com multiplos agentes no CrewAI.

Principais pontos consolidados:
- especializacao de papeis melhora foco e qualidade das analises;
- tools de leitura, busca e testes aumentam rastreabilidade;
- tarefas sequenciais ajudam a manter o contexto e reduzir lacunas entre etapas;
- revisao final explicita riscos e condicoes para avancar com seguranca.

Resultado final esperado: uma proposta tecnica fundamentada em evidencias, pronta para validacao humana antes de qualquer alteracao no codigo.